In [5]:
import zipfile                                                   # read files straight out of a zip, no extracting
import pandas as pd

z = zipfile.ZipFile("../data/raw/stroebl/TP_k01_02__47627056.zip")   # cell S002 (stage 1)
df = pd.read_csv(z.open("TP_k01_02/TP_k01_02_02_CU.csv"))        # its first check-up

df["t"] = pd.to_timedelta(df["run_time"]).dt.total_seconds()     # "HH:MM:SS.sss" → seconds, so we can do arithmetic
df["dt"] = df["t"].diff().fillna(0)                              # time since the previous row, in seconds
df["step"] = (df["step_type"] != df["step_type"].shift()).cumsum()   # new step number each time step_type changes

steps = df.groupby("step").agg(                                  # one summary row per step
    type=("step_type", "first"),                                 # its step_type code
    minutes=("dt", lambda s: s.sum() / 60),                      # how long it lasted
    mean_A=("c_cur", "mean"),                                    # average current (negative = discharging)
    V_start=("c_vol", "first"), V_end=("c_vol", "last"),         # voltage at start and end
)
steps["Ah"] = df.groupby("step").apply(lambda g: (g["c_cur"] * g["dt"]).sum() / 3600)   # charge moved: ∫ I dt, seconds → hours
steps.round(3)

,type,minutes,mean_A,V_start,V_end,Ah
step,,,,,,
1,0,221.588,-0.088,3.267,2.760,-0.284
2,21,140.211,2.080,2.761,4.199,4.875
3,0,25.004,0.000,4.181,4.166,0.000
4,22,63.599,-4.571,4.149,2.501,-4.889
5,0,109.904,1.681,2.511,3.716,2.443
6,5012,0.504,-1.632,3.716,3.682,-0.014
7,5014,0.504,0.000,3.696,3.720,0.000
8,0,10.008,1.568,3.711,3.728,0.014
9,5011,0.506,1.629,3.717,3.783,0.014


In [17]:
import re
from pathlib import Path

def parse_date(s):
    """Parse dd.mm.yyyy or dd.mm.yy; return NaT (missing) if the date has a typo."""
    for fmt in ("%d.%m.%Y", "%d.%m.%y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            pass
    return pd.NaT                                                # unparseable, e.g. "2.12.233"

issues = []                                                      # global log: one entry per skipped or odd test

def checkup_capacities(zip_path):
    zip_path = Path(zip_path)
    z = zipfile.ZipFile(zip_path)
    names = set(z.namelist())                                    # for fast "does this file exist?" checks
    rows, serial = [], None
    for name in sorted(names):
        m = re.search(r"_(\d+)_(CU|exCU|ET_T23|AT_T23)\.csv$", name)
        if not m:
            continue
        meta_name = name.replace(".csv", "_meta.txt")
        meta = z.read(meta_name).decode("latin-1") if meta_name in names else ""   # empty string if the meta file is missing
        s = re.search(r"Internal cell serial: (\S+)", meta)
        serial = s.group(1) if s else serial                     # if missing, reuse the serial from this cell's other tests
        d_ = re.search(r"Measurement start date: (\S+)", meta)
        date = parse_date(d_.group(1)) if d_ else pd.NaT
        if not meta or not s or pd.isna(date):
            issues.append((zip_path.name, name.split("/")[-1], "missing meta / serial / bad date"))   # log, but keep going
        d = pd.read_csv(z.open(name), usecols=["run_time", "c_cur", "step_type"])
        d["dt"] = pd.to_timedelta(d["run_time"]).dt.total_seconds().diff().fillna(0)
        dis = d[d["step_type"] == 22]
        rows.append({"serial": serial, "zip": zip_path.name,     # also keep the zip name, to trace problems back
                     "cu": int(m.group(1)), "kind": m.group(2), "n_dis_rows": len(dis),
                     "date": date, "Q_Ah": -(dis["c_cur"] * dis["dt"]).sum() / 3600})
    out = pd.DataFrame(rows)
    out["serial"] = out["serial"].ffill().bfill()                # fill a missing serial from this cell's other rows
    out["days"] = (out["date"] - out["date"].min()).dt.days     # days since this cell's first test (NaN where the date is missing)
    return out

s002 = checkup_capacities("../data/raw/stroebl/TP_k01_02__47627056.zip")
s002

,serial,zip,cu,kind,n_dis_rows,date,Q_Ah,days
0,S002,TP_k01_02__47627056.zip,1,ET_T23,388,2021-09-15,4.874845,0
1,S002,TP_k01_02__47627056.zip,2,CU,389,2021-10-26,4.888621,41
2,S002,TP_k01_02__47627056.zip,3,exCU,388,2021-11-30,4.886363,76
3,S002,TP_k01_02__47627056.zip,4,CU,386,2022-01-18,4.876081,125
4,S002,TP_k01_02__47627056.zip,5,CU,386,2022-03-15,4.877537,181
5,S002,TP_k01_02__47627056.zip,6,CU,384,2022-05-10,4.865900,237
6,S002,TP_k01_02__47627056.zip,7,exCU,385,2022-06-05,4.861765,263
7,S002,TP_k01_02__47627056.zip,8,CU,385,2022-08-30,4.856918,349
8,S002,TP_k01_02__47627056.zip,9,AT_T23,383,2022-09-12,4.815369,362


In [18]:
# parse the cells
from pathlib import Path
import pandas as pd
RAW = Path("../data/raw/stroebl")
zips = sorted(RAW.glob("TP_*.zip"))                              # all 279 cell zips
print(len(zips), "zips")

parts, failed = [], []
for i, zp in enumerate(zips):
    try:
        parts.append(checkup_capacities(zp))                     # one small table per cell
    except Exception as e:
        failed.append((zp.name, repr(e)))                        # record failures instead of stopping the loop
    if i % 25 == 0:
        print(i, zp.name)                                        # progress, about every 25 files

cap = pd.concat(parts, ignore_index=True)                        # stack all cells into one long table

meta = pd.read_csv(RAW / "experiments_meta.csv", dtype={"serial_internal": str})   # read the serial as text so "167" and "S002" match
meta.columns = meta.columns.str.strip()
meta = meta.rename(columns={"serial": "file_name"})              # metadata's "serial" is the file name (TP_k01_02); rename it to avoid a clash with our cell serial
cap = cap.merge(meta, left_on="serial", right_on="serial_internal", how="left")   # attach test point, stage, stress factors to every row

Path("../data/processed").mkdir(exist_ok=True)
cap.to_parquet("../data/processed/capacity_rpt.parquet")         # save; reloads in under a second next time

print("cells parsed:", cap["serial"].nunique(), "| rows:", len(cap), "| failed:", len(failed))
print("rows with no metadata match:", cap["tp"].isna().sum())    # should be 0
print("tests with no step-22 discharge:", (cap["n_dis_rows"] == 0).sum())   # should be 0
failed[:5]

279 zips
0 TP_k01_01__47627050.zip
25 TP_k03_01__47627626.zip
50 TP_k05_02__47628238.zip
75 TP_k07_02__47630674.zip
100 TP_k09_03__47628352.zip
125 TP_k11_03__47630776.zip
150 TP_z04_01__47629486.zip
175 TP_z08_01__47632615.zip
200 TP_z12_02__47632411.zip
225 TP_z16_02__47632672.zip
250 TP_z20_03__47630290.zip
275 TP_z24_03__47632786.zip
cells parsed: 270 | rows: 3082 | failed: 0
rows with no metadata match: 24
tests with no step-22 discharge: 4


[]

In [19]:
pd.Series([e.split("(")[0] + ": " + e[:80] for _, e in failed]).value_counts()   # count failures by error type

Series([], Name: count, dtype: int64)

In [20]:
per_zip = cap.groupby("zip")["serial"].agg(lambda s: sorted(set(s)))           # serial(s) found in each zip
dups = cap.groupby("serial")["zip"].nunique().loc[lambda s: s > 1]              # serials that appear in more than one zip
print("serials shared by >1 zip:\n", cap[cap["serial"].isin(dups.index)].groupby("serial")["zip"].unique(), "\n")
print("serials with no metadata match:\n", cap.loc[cap["tp"].isna()].groupby(["zip", "serial"]).size())

serials shared by >1 zip:
 serial
INT003    [TP_k04_03__47628247.zip, TP_k07_01__47630437....
INT004    [TP_k04_02__47630371.zip, TP_k05_01__47628256....
INT005    [TP_k05_02__47628238.zip, TP_z10_02__47632315....
INT007    [TP_k06_01__47628271.zip, TP_z06_02__47632783....
INT009    [TP_k06_01__47630443.zip, TP_k06_03__47628274....
INT011    [TP_z04_02__47630293.zip, TP_z11_03__47632756....
INT018    [TP_z06_01__47632657.zip, TP_z06_03__47632534....
INT020    [TP_z07_02__47632486.zip, TP_z09_03__47632741....
INT027    [TP_k05_05__47629804.zip, TP_z09_03__47632327....
INT029    [TP_z04_03__47639014.zip, TP_z10_02__47632324....
INT031    [TP_z09_01__47632780.zip, TP_z11_01__47632357....
INT037    [TP_z04_02__47632810.zip, TP_z13_01__47632228....
INT038    [TP_z13_02__47632366.zip, TP_z13_03__47632408....
INT043    [TP_k06_04__47629813.zip, TP_z15_01__47632531....
INT044    [TP_z11_02__47632774.zip, TP_z15_02__47632516....
INT045    [TP_k07_02__47630674.zip, TP_z15_03__47632441....
INT051

In [21]:
first = cap.groupby("zip")["date"].min().rename("start").reset_index()        # each zip's first test date
first["file_name"] = first["zip"].str.split("__").str[0]                       # "TP_k01_02__47627056.zip" → "TP_k01_02"
print(first["start"].dt.to_period("M").value_counts().sort_index())            # how many zips started in each month

start
2021-09    51
2021-10    90
2022-12    39
2023-01    27
2023-02    72
Freq: M, Name: count, dtype: int64


In [22]:
first["stage"] = (first["start"] >= "2022-06-01").astype(int) + 1             # 1 if started in 2021, 2 if started from Dec 2022

meta = pd.read_csv(RAW / "experiments_meta.csv", dtype={"serial_internal": str})
meta.columns = meta.columns.str.strip()
meta = meta.rename(columns={"serial": "file_name"})

link = first.merge(meta, on=["file_name", "stage"], how="left", validate="one_to_one")   # validate: errors if any (file_name, stage) pair matches more than one row
print("zips:", len(link), "| unmatched:", link["serial_internal"].isna().sum(),
      "| unique cells:", link["serial_internal"].nunique())              # want: 279 | 0 | 279

base = cap[["zip", "cu", "kind", "n_dis_rows", "date", "Q_Ah", "days"]]   # the measurements only; drop the old, unreliable serial-based columns
cap = base.merge(link.drop(columns=["start"]), on="zip", how="left")       # attach the correct metadata to every test
cap = cap.rename(columns={"serial_internal": "cell_id"})                   # the real unique cell ID from now on

cap.to_parquet("../data/processed/capacity_rpt.parquet")
print("rows:", len(cap), "| cells:", cap["cell_id"].nunique())

zips: 279 | unmatched: 0 | unique cells: 261
rows: 3082 | cells: 261


In [23]:
cap["cell_id"] = cap["file_name"] + "_s" + cap["stage"].astype(str)        # e.g. "TP_k01_02_s1": unique by construction
print("cells:", cap["cell_id"].nunique(), "| rows:", len(cap))             # want 279
print("authors' serial_internal unique values:", meta["serial_internal"].nunique(), "of", len(meta))   # confirms the reuse is in their metadata
cap.to_parquet("../data/processed/capacity_rpt.parquet")

cells: 279 | rows: 3082
authors' serial_internal unique values: 261 of 279
